# Monte Carlo simulation

**Curriculum:** advanced quant track.

**Prerequisites:** work through `01_foundations/core_types_and_money.ipynb` and `01_foundations/math_toolkit.ipynb`. This notebook assumes comfort with `Money`-like results and basic probability.

**In this notebook:** Monte Carlo option pricing in `finstack_quant.models.monte_carlo` — analytical Black–Scholes benchmarks, canonical European, path-dependent, and American (LSMC) pricers, and an ATM European comparison.


## Monte Carlo option pricing

Under **risk-neutral** pricing, a European derivative value is the discounted expectation of its payoff under a model for the underlying. When no closed form exists, we **simulate** many paths of the process, average payoffs, and discount — the **Monte Carlo** estimator.

For a European call on a stock following geometric Brownian motion (GBM), the Black–Scholes formula gives the exact benchmark. Monte Carlo should agree within simulation error (stderr shrinks like \(1/\sqrt{N}\) for \(N\) paths).

`finstack_quant.models.monte_carlo` exposes:

- **Analytical** `models.bs_price` (`is_call=True/False`) for sanity checks.
- **High-level pricing** through `EuropeanPricer`.
- **Exotics:** `PathDependentPricer` (e.g. Asian), `LsmcPricer` (American via Longstaff–Schwartz).

Process and payoff parameters (rate, volatility, strike, etc.) are passed **directly** as numeric arguments to the pricer constructors and methods — there are no standalone Python parameter-object classes.


### Black–Scholes (analytical benchmark)

Use these closed-form prices to validate Monte Carlo output and to check **put–call parity**: \(C - P = e^{-rT}(F - K)\) with forward \(F = S e^{(r-q)T}\).


In [ ]:
from finstack_quant.models import bs_price
import math

from finstack_quant.models.monte_carlo import (
    EuropeanPricer,
    LsmcPricer,
    PathDependentPricer,
)

spot = strike = 100.0
r = 0.05
T = 1.0
bs_call = bs_price(spot, strike, r, 0.0, 0.20, T, True)
bs_put = bs_price(spot, strike, r, 0.0, 0.20, T, False)
disc = math.exp(-r * T)
print(f"BS Call: {bs_call:.6f}")
print(f"BS Put: {bs_put:.6f}")
print(f"Put-Call Parity: C-P = {bs_call - bs_put:.6f}, S-K*e^(-rT) = {spot - strike * disc:.6f}")


### `EuropeanPricer` (single-step-style European MC)

`EuropeanPricer` runs a straightforward GBM simulation and returns a `MoneyEstimate`: mean estimate in `mean` (a `Money`-like amount + currency), standard error, asymptotic confidence band, and path count.


In [ ]:
pricer = EuropeanPricer(num_paths=50_000, seed=42)
result = pricer.price_call(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"MC Call: {result.mean.amount:.6f}")
print(f"Currency: {result.mean.currency.code}")
print(f"Std error: {result.stderr:.6f}")
print(f"95% CI: [{result.ci_lower.amount:.6f}, {result.ci_upper.amount:.6f}]")
print(f"Num paths: {result.num_paths}")

result_put = pricer.price_put(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"MC Put: {result_put.mean.amount:.6f}")


### Stochastic processes

The Python pricers fix the underlying dynamics to **geometric Brownian motion (GBM)** and accept risk-neutral drift parameters (``rate``, ``div_yield``, ``vol``) directly as numeric arguments. Alternative processes (Heston, CIR, Schwartz–Smith, …) are implemented in the Rust crate `finstack-quant-models` and can be consumed from Rust or via custom extensions; they are not surfaced as Python classes today.

A few process-level helpers *are* bound as functions — for example `heston_satisfies_feller`, which validates Heston parameters and tests the **Feller condition** \(2\kappa\theta > \xi^2\). When that condition holds, the variance process stays strictly positive; when it fails, discretization schemes must handle variance hitting zero. See `monte_carlo/stochastic_processes.ipynb` for the full process tour.


In [ ]:
from finstack_quant.models.monte_carlo import heston_satisfies_feller

gbm_params = dict(rate=0.05, div_yield=0.0, vol=0.20)
print(f"GBM params: {gbm_params}")

heston_params = dict(
    rate=0.05, div_yield=0.0, v0=0.04,
    kappa=2.0, theta=0.04, xi=0.3, rho=-0.7,
)
# The Feller condition 2*kappa*theta > xi^2 keeps the Heston variance process
# strictly positive. Ask the binding rather than re-deriving it: it validates the
# parameters (positive kappa/theta/vol-of-vol) as well as testing the inequality.
feller_ok = heston_satisfies_feller(
    heston_params["kappa"], heston_params["theta"], heston_params["xi"],
)
print(f"Heston params: {heston_params}")
print(f"Feller satisfied (2*kappa*theta > xi^2): {feller_ok}")

### Path-dependent: arithmetic Asian call

`PathDependentPricer` prices payoffs that depend on the whole path; here an **arithmetic average** Asian call with fixings at each simulated step.


In [ ]:
asian_pricer = PathDependentPricer(num_paths=10_000, seed=42)
asian_result = asian_pricer.price_asian_call(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0,
    vol=0.20, expiry=1.0, num_steps=252,
)
print(f"Asian Call: {asian_result.mean.amount:.6f}")


### American options: `LsmcPricer`

**Longstaff–Schwartz Monte Carlo** estimates continuation value via regression on basis functions. In continuous-time theory, an American put is **at least** as valuable as the European with the same terms; LSMC is a biased, noisy estimator, so compare against both **BS** and a **European MC** control with the same seed/paths when validating.

**Exercise grid bias:** `num_steps` fixes the Bermudan exercise grid. Too few steps **coarsens** the exercise boundary versus continuous exercise (often **biasing low** early-exercise value); using a **252-step** (roughly daily) grid is a common teaching default, though runtime grows with paths × steps.


In [ ]:
from finstack_quant.models import bs_price
lsmc = LsmcPricer(num_paths=10_000, seed=42, num_steps=252)
am_put = lsmc.price_american_put(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0,
    vol=0.30, expiry=1.0,
)
bs_put_ref = bs_price(100.0, 100.0, 0.05, 0.0, 0.30, 1.0, False)
mc_euro_put = EuropeanPricer(num_paths=10_000, seed=42).price_put(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.30, expiry=1.0,
)
print(f"American Put (LSMC): {am_put.mean.amount:.6f}")
print(f"European Put (BS):   {bs_put_ref:.6f}")
print(f"European Put (MC):   {mc_euro_put.mean.amount:.6f}")
print(f"LSMC >= European MC: {am_put.mean.amount >= mc_euro_put.mean.amount}")
print("(If False, treat as a signal to raise num_steps/num_paths or audit LSMC settings.)")


## Mini-example: one ATM call, two estimators

Compare the Black–Scholes analytical result with `EuropeanPricer` using 50,000 paths. The table reports price, error versus Black–Scholes, and the Monte Carlo confidence interval.


In [ ]:
from finstack_quant.models import bs_price
spot = strike = 100.0
rate = 0.05
div_yield = 0.0
vol = 0.20
expiry = 1.0
num_paths = 50_000
seed = 42

bs = bs_price(spot, strike, rate, div_yield, vol, expiry, True)

ep = EuropeanPricer(num_paths=num_paths, seed=seed)
r_ep = ep.price_call(
    spot=spot, strike=strike, rate=rate, div_yield=div_yield, vol=vol, expiry=expiry,
)


def fmt_ci(r):
    return f"[{r.ci_lower.amount:.6f}, {r.ci_upper.amount:.6f}]"

rows = [
    ("Black-Scholes (exact)", bs, 0.0, "n/a (exact)"),
    ("EuropeanPricer (50k)", r_ep.mean.amount, r_ep.mean.amount - bs, fmt_ci(r_ep)),
]

print(f"Parameters: S={spot}, K={strike}, r={rate}, q={div_yield}, sigma={vol}, T={expiry}y")
print()
w0, w1, w2, w3 = 28, 14, 14, 36
print(f"{'Method':<{w0}} {'Price':>{w1}} {'Err vs BS':>{w2}} {'95% CI':<{w3}}")
print("-" * (w0 + w1 + w2 + w3))
for name, px, err, ci in rows:
    print(f"{name:<{w0}} {px:>{w1}.6f} {err:>{w2}.6f} {ci:<{w3}}")


## Takeaways

- **Monte Carlo** estimates discounted risk-neutral expectations; stderr scales with \(1/\sqrt{N}\) — increase paths or use variance reduction when tight CIs matter.
- **Black–Scholes** Europeans on GBM are the standard benchmark; MC means should land inside the reported interval around the analytical value.
- **`EuropeanPricer`** is the canonical public Python entry point for GBM European pricing.
- **Path-dependent** and **American** payoffs need path-wise or backward schemes (`PathDependentPricer`, `LsmcPricer`); always cross-check economics (e.g. American \(\geq\) European).
- **Next steps:** combine with curves and calendars from the foundations notebooks for full term-structure setups, or explore `HestonProcess` and other processes for richer dynamics.


### Named Monte Carlo estimate types

Currency-valued pricers return `MoneyEstimate`; scalar Monte Carlo routines expose the sibling `Estimate` type for non-money outputs.

In [ ]:
from finstack_quant.models.monte_carlo import Estimate, MoneyEstimate

print("MoneyEstimate type:", MoneyEstimate.__name__)
print("call result typed:", isinstance(result, MoneyEstimate))
print("Scalar estimate type available:", Estimate.__name__)